# Simple bridge: encoder + LoRA → bridge → Gemma + LoRA, trained end to end

```
"Don: I don't trust someone who changes their story every time."
        │
        ▼
ModernBERT-base + LoRA ──► bridge: Linear 768 → 2560 + RMSNorm                                   [GPU 0]
                                  │ memory
                                  ▼
Gemma 3 4B IT (4-bit) + LoRA                                                                       [GPU 1]
   input: only the start token <bos>; cross-attention after every 4th layer reads the memory
   output: "Jo har baar kahani badalta hai, uspe bharosa nahi. Simple."  <end_of_turn>
```

- **Everything goes through the encoder:** persona and sentence together are the encoder's input. **The decoder receives nothing but the bridge**: its own input is a single start token, then the tokens it generates.
- **Decoder:** `google/gemma-3-4b-it` in 4-bit (NF4, fp32 compute), no other adapter. No chat template and no prompt: the decoder starts from `<bos>` and the reply ends with `<end_of_turn>`, the token Gemma already uses to finish a reply.
- **After training there are two LoRAs** (encoder and decoder) plus the bridge and cross-attention; base weights stay frozen.
- **Training:** cross-entropy between the decoder's predictions and the reference translation, all five parts trained together: encoder LoRA, bridge, cross-attention (with gates starting at 0), decoder LoRA. Base weights frozen.
- **Data:** v4 merged — 768 training pairs; checkpoint chosen on validation loss (75); test (75) and out-of-domain (114) translated once at the end.

In [ ]:
# Cell 0 — Hugging Face token (Gemma 3 is gated). Clear it before saving or sharing the notebook.

HF_TOKEN = ""

## Cell 1 — Setup

Two GPUs (Kaggle: *GPU T4 × 2*): encoder and bridge on `cuda:0`, decoder and cross-attention on `cuda:1`; fp32 compute, 4-bit decoder weights.

In [2]:
# Cell 1 — Packages, settings, pinned revisions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2" -U bitsandbytes

import json, math, random, statistics, time, urllib.request
from collections import Counter

import peft
import torch
import torch.nn as nn
import transformers
from huggingface_hub import HfApi, login

ENCODER, DECODER = "answerdotai/ModernBERT-base", "google/gemma-3-4b-it"
DATA = ("https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/"
        "project4-hindi-decoder/data/persona_translation_v4/merged/")
XATTN_EVERY, XATTN_HEADS = 4, 8
MAX_NEW_TOKENS = 64
SEED = 42
DTYPE = torch.float32

assert torch.cuda.device_count() >= 2, "this notebook needs two GPUs (Kaggle: GPU T4 x2)"
ENC_DEVICE, DEC_DEVICE = torch.device("cuda:0"), torch.device("cuda:1")
HF_TOKEN = HF_TOKEN.strip() or None
assert HF_TOKEN, "Gemma 3 is gated: paste a Hugging Face token in Cell 0"
login(HF_TOKEN)
REV = {r: HfApi().model_info(r, token=HF_TOKEN).sha for r in (ENCODER, DECODER)}
print({"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
       "gpus": [torch.cuda.get_device_name(i) for i in range(2)], "revisions": REV})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 91.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.8 MB/s eta 0:00:00:00:0100:01
{'torch': '2.11.0+cu128', 'transformers': '5.17.0', 'peft': '0.21.2', 'gpus': ['Tesla T4', 'Tesla T4'], 'revisions': {'answerdotai/ModernBERT-base': '8949b909ec900327062f0ebf497f51aef5e6f0c8', 'google/gemma-3-4b-it': '093f9f388b31de276ce2de164bdc2081324b9767'}}


## Cell 2 — Data: encoder input and decoder target

Encoder input: `"<persona name>: <English sentence>"`. Decoder target: the reference Hinglish line.

In [3]:
# Cell 2 — Load v4 merged; build (encoder input, target) records

def fetch_jsonl(name):
    return [json.loads(l) for l in urllib.request.urlopen(DATA + name).read().decode("utf-8").splitlines() if l]


MANIFEST = json.load(urllib.request.urlopen(DATA + "manifest.json"))
NAME = {p: v["name"] for p, v in MANIFEST["personas"].items()}


def record(row):
    return {**row, "persona": row["persona_id"], "enc_input": f"{NAME[row['persona_id']]}: {row['source']}"}


SPLITS = {s: [record(r) for r in fetch_jsonl(f"{s}.jsonl")] for s in ("train", "validation", "test", "ood")}
TRAIN, VAL = SPLITS["train"], SPLITS["validation"]
print({s: len(rs) for s, rs in SPLITS.items()})
print("encoder input:", TRAIN[0]["enc_input"])
print("target:       ", TRAIN[0]["target"])

{'train': 768, 'validation': 75, 'test': 75, 'ood': 114}
encoder input: Don: I don't trust someone who changes their story every time.
target:        Jo har baar kahani badalta hai, uspe bharosa nahi. Simple.


## Cell 3 — Models: ModernBERT (GPU 0); Gemma 3 4B IT in 4-bit (GPU 1)

In [4]:
# Cell 3 — Load the encoder and the decoder

from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

used = {d: torch.cuda.memory_allocated(d) / 2**30 for d in (ENC_DEVICE, DEC_DEVICE)}
if max(used.values()) > 0.5:
    raise RuntimeError("GPU memory already in use: restart the kernel and run all cells once, from the top.")

enc_tok = AutoTokenizer.from_pretrained(ENCODER, revision=REV[ENCODER])
enc_cfg = AutoConfig.from_pretrained(ENCODER, revision=REV[ENCODER])
if hasattr(enc_cfg, "reference_compile"):
    enc_cfg.reference_compile = False
encoder = AutoModel.from_pretrained(ENCODER, revision=REV[ENCODER], config=enc_cfg, dtype=DTYPE,
                                    attn_implementation="sdpa").to(ENC_DEVICE)

dec_tok = AutoTokenizer.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
decoder = Gemma3ForConditionalGeneration.from_pretrained(DECODER, revision=REV[DECODER], dtype=DTYPE, token=HF_TOKEN,
                                                         quantization_config=quant, device_map={"": DEC_DEVICE.index},
                                                         attn_implementation="eager")
START = dec_tok.bos_token_id                  # the decoder's only input token
EOS = dec_tok.convert_tokens_to_ids("<end_of_turn>")
eos = decoder.generation_config.eos_token_id
STOPS = sorted(set((eos if isinstance(eos, list) else [eos]) + [EOS]))
PAD = dec_tok.pad_token_id
D_ENC, D_DEC = encoder.config.hidden_size, decoder.config.text_config.hidden_size
LAYERS = next(m for n, m in decoder.named_modules() if n.endswith("language_model.layers"))
N_LAYERS = len(LAYERS)
print(f"encoder width {D_ENC} | decoder width {D_DEC}, {N_LAYERS} layers | start {START}, end {EOS}, stops {STOPS} | "
      f"GPU 0 {torch.cuda.memory_allocated(ENC_DEVICE) / 2**30:.1f} GB, GPU 1 {torch.cuda.memory_allocated(DEC_DEVICE) / 2**30:.1f} GB")

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

encoder width 768 | decoder width 2560, 34 layers | start 2, end 106, stops [1, 106] | GPU 0 0.6 GB, GPU 1 4.5 GB


## Cell 4 — Bridge and gated cross-attention

Bridge: `Linear(768 → 2560)` + RMSNorm. One cross-attention block after every 4th decoder layer (queries from the decoder, keys and values from the memory, encoder padding masked), added through `tanh(gate)` with the gate starting at 0. Attached with forward hooks, so the decoder's own modules are unchanged.

In [5]:
# Cell 4 — Bridge, cross-attention blocks, hooks

class Bridge(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.proj = nn.Linear(d_in, d_out, bias=False)
        self.norm = nn.RMSNorm(d_out)

    def forward(self, h):
        return self.norm(self.proj(h))


class GatedCrossAttention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.norm = nn.RMSNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True, bias=False)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, h, memory, pad):
        out, _ = self.attn(self.norm(h), memory, memory, key_padding_mask=pad, need_weights=False)
        return h + torch.tanh(self.gate) * out


class Memory:
    memory, pad = None, None


MEM = Memory()
XATTN_LAYERS = list(range(XATTN_EVERY - 1, N_LAYERS, XATTN_EVERY))
torch.manual_seed(SEED)
bridge = Bridge(D_ENC, D_DEC).to(ENC_DEVICE)
xattn = nn.ModuleDict({str(i): GatedCrossAttention(D_DEC, XATTN_HEADS) for i in XATTN_LAYERS}).to(DEC_DEVICE)


def make_hook(block):
    def hook(module, args, output):
        h = output[0] if isinstance(output, tuple) else output
        h = block(h, MEM.memory, MEM.pad)
        return (h, *output[1:]) if isinstance(output, tuple) else h
    return hook


HOOKS = [LAYERS[i].register_forward_hook(make_hook(xattn[str(i)])) for i in XATTN_LAYERS]
print(f"cross-attention after decoder layers {XATTN_LAYERS}")

cross-attention after decoder layers [3, 7, 11, 15, 19, 23, 27, 31]


## Cell 5 — LoRA at both ends; the forward pass

- Encoder LoRA on ModernBERT's `Wqkv`, `Wo`, `Wi`; decoder LoRA on Gemma's attention and MLP projections (language model only); rank 16. `prepare_model_for_kbit_training` sets up the 4-bit decoder (gradient checkpointing).
- **Forward pass:** encoder input → encoder → bridge → memory; decoder input = `<bos> + target`, labels = target + `<end_of_turn>`; the loss is the decoder's cross-entropy against the reference.

In [6]:
# Cell 5 — LoRA; the encode / loss / generate functions

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

decoder = prepare_model_for_kbit_training(decoder, use_gradient_checkpointing=True,
                                          gradient_checkpointing_kwargs={"use_reentrant": False})
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
dec_model = get_peft_model(decoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=rf".*language_model.*\.({'|'.join(PROJ)})"))
encoder = get_peft_model(encoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                             target_modules=["Wqkv", "Wo", "Wi"]))
GROUPS = {"encoder LoRA": [p for n, p in encoder.named_parameters() if "lora_" in n],
          "bridge": list(bridge.parameters()),
          "cross-attention": [p for n, p in xattn.named_parameters() if not n.endswith("gate")],
          "gates": [b.gate for b in xattn.values()],
          "decoder LoRA": [p for n, p in dec_model.named_parameters() if "lora_" in n]}
for name, params in GROUPS.items():
    for p in params:
        p.requires_grad_(True)
    print(f"  {name:16s} {sum(p.numel() for p in params) / 1e6:7.2f} M trainable")


def encode(texts):
    t = enc_tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(ENC_DEVICE)
    h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
    MEM.memory, MEM.pad = bridge(h).to(DEC_DEVICE), (t["attention_mask"] == 0).to(DEC_DEVICE)


def loss_on(records):
    encode([r["enc_input"] for r in records])
    seqs = [[START] + dec_tok(r["target"], add_special_tokens=False)["input_ids"] + [EOS] for r in records]
    T = max(map(len, seqs))
    ids = torch.full((len(seqs), T), PAD)
    labels = torch.full((len(seqs), T), -100)
    mask = torch.zeros((len(seqs), T), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)], labels[i, 1:len(s)], mask[i, :len(s)] = torch.tensor(s), torch.tensor(s[1:]), 1
    return dec_model(input_ids=ids.to(DEC_DEVICE), attention_mask=mask.to(DEC_DEVICE), labels=labels.to(DEC_DEVICE),
                     token_type_ids=torch.zeros_like(ids).to(DEC_DEVICE)).loss


@torch.no_grad()
def translate(r):
    dec_model.eval()
    encoder.eval()
    encode([r["enc_input"]])
    start = torch.tensor([[START]], device=DEC_DEVICE)
    out = dec_model.generate(input_ids=start, attention_mask=torch.ones_like(start), do_sample=False, num_beams=1,
                             max_new_tokens=MAX_NEW_TOKENS, eos_token_id=STOPS, pad_token_id=PAD,
                             temperature=None, top_p=None, top_k=None)
    new = out[0, 1:].tolist()
    return {"output": dec_tok.decode(new, skip_special_tokens=True).strip(), "stopped": bool(new) and new[-1] in STOPS}


@torch.no_grad()
def validation_loss(batch=8):
    dec_model.eval()
    encoder.eval()
    total = 0.0
    for i in range(0, len(VAL), batch):
        chunk = VAL[i:i + batch]
        total += loss_on(chunk).item() * len(chunk)
    return round(total / len(VAL), 4)


with torch.no_grad():                         # with the gates at 0 the decoder is plain Gemma, from <bos> only
    gates0 = translate(VAL[0])["output"]
print("before training (gates at 0):", gates0[:120])

  encoder LoRA        3.38 M trainable
  bridge              1.97 M trainable
  cross-attention   209.74 M trainable
  gates               0.00 M trainable
  decoder LoRA       29.80 M trainable
before training (gates at 0): The 2023-2024 NBA season is in full swing, and the league is brimming with exciting storylines, star power, and unexpect


## Cell 6 — Train everything together

Micro-batches of 4 with 2-step accumulation (8 pairs per update); AdamW with linear decay; learning rates: encoder LoRA 1e-4, bridge and cross-attention 3e-4, gates 1e-2, decoder LoRA 5e-5 (Gemma over-fitted at 1e-4 in Phase 1). Up to 8 epochs, early stopping after 2 epochs without a lower validation loss; the best epoch is kept. After each epoch: training and validation loss, and three sample translations.

In [7]:
# Cell 6 — Training loop

MAX_EPOCHS, PATIENCE, MICRO, ACCUM = 8, 2, 4, 2
LRS = {"encoder LoRA": 1e-4, "bridge": 3e-4, "cross-attention": 3e-4, "gates": 1e-2, "decoder LoRA": 5e-5}
opt = torch.optim.AdamW([{"params": GROUPS[g], "lr": lr} for g, lr in LRS.items()], weight_decay=0.0)
total_steps = MAX_EPOCHS * -(-len(TRAIN) // (MICRO * ACCUM))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1 - s / total_steps)
SAMPLES = [VAL[0], VAL[4], VAL[8]]


def snapshot():
    return [p.detach().to("cpu", copy=True) for params in GROUPS.values() for p in params]


def restore(state):
    for p, s in zip([p for params in GROUPS.values() for p in params], state):
        p.data.copy_(s.to(p.device))


LOG = [{"epoch": 0, "validation_loss": validation_loss()}]
print(LOG[-1])
best = (LOG[0]["validation_loss"], 0, snapshot())
since_best, rng, t0 = 0, random.Random(SEED), time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    dec_model.train()
    encoder.train()
    order = rng.sample(TRAIN, len(TRAIN))
    batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
    run = []
    for n, b in enumerate(batches, 1):
        loss = loss_on(b)
        (loss / ACCUM).backward()
        run.append(loss.item())
        if n % ACCUM == 0 or n == len(batches):
            opt.step()
            sched.step()
            opt.zero_grad()
    LOG.append({"epoch": epoch, "train_loss": round(sum(run) / len(run), 4), "validation_loss": validation_loss(),
                "gates": [round(math.tanh(b.gate.item()), 3) for b in xattn.values()]})
    print(LOG[-1], f"| {time.perf_counter() - t0:.0f}s")
    for r in SAMPLES:
        print(f"    {r['enc_input'][:70]:70s} -> {translate(r)['output']}")
    if LOG[-1]["validation_loss"] < best[0]:
        best, since_best = (LOG[-1]["validation_loss"], epoch, snapshot()), 0
    else:
        since_best += 1
    if since_best >= PATIENCE:
        print(f"early stop: no lower validation loss for {PATIENCE} epochs")
        break

restore(best[2])
BEST_EPOCH = best[1]
print(f"kept epoch {BEST_EPOCH} (validation loss {best[0]}) | peak GPU 1 "
      f"{torch.cuda.max_memory_allocated(DEC_DEVICE) / 2**30:.1f} GB")

{'epoch': 0, 'validation_loss': 7.3383}


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


{'epoch': 1, 'train_loss': 4.3231, 'validation_loss': 4.0494, 'gates': [-0.043, -0.07, -0.07, -0.102, 0.085, 0.109, -0.105, 0.272]} | 269s
    Don: We should celebrate the progress without ignoring the remaining w -> Dono ka meeting 10 baje se pehle.
    Anthony Gonsalves: I will support the decision once everyone understan -> Dono ka meeting 10 baje se pehle, 11 baje baad.
    Jai (Sholay): Don't confuse being busy with making progress.           -> Pehle ek baar check karo.
{'epoch': 2, 'train_loss': 2.8165, 'validation_loss': 4.0234, 'gates': [-0.04, -0.066, -0.081, -0.119, 0.096, 0.125, -0.12, 0.253]} | 551s
    Don: We should celebrate the progress without ignoring the remaining w -> Pehle sabki suno. Phir decide karo.
    Anthony Gonsalves: I will support the decision once everyone understan -> Pehle check karna hoga, phir approve karna hoga.
    Jai (Sholay): Don't confuse being busy with making progress.           -> Pehle sabki suno. Phir decide karo.
{'epoch': 3, 'train_loss'

## Cell 7 — Translate validation, test and out-of-domain; scores

chrF against the best of the references (character n-grams 1–6, β = 2), share of outputs that stopped on `<end_of_turn>`, and share that are a word-for-word copy of some training target (a sign of memorising instead of translating).

In [8]:
# Cell 7 — Translations and scores

def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, q = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * q / (beta ** 2 * p + q) if p + q else 0.0


train_targets = {r["target"].strip() for r in TRAIN}
RESULTS, t0 = [], time.perf_counter()
for split in ("validation", "test", "ood"):
    for r in SPLITS[split]:
        out = translate(r)
        RESULTS.append({"split": split, "id": r["id"], "persona": r["persona"], "source": r["source"],
                        "references": r["references"], **out,
                        "chrf": max(chrf(out["output"], ref) for ref in r["references"])})
print(f"{len(RESULTS)} translations in {time.perf_counter() - t0:.0f}s")
SCORES = {}
for split in ("validation", "test", "ood"):
    rs = [x for x in RESULTS if x["split"] == split]
    SCORES[split] = {"n": len(rs), "chrf": round(statistics.mean(x["chrf"] for x in rs), 1),
                     "stopped": round(sum(x["stopped"] for x in rs) / len(rs), 3),
                     "distinct_outputs": len({x["output"] for x in rs}),
                     "copies_of_training_targets": sum(x["output"] in train_targets for x in rs)}
    print(split, SCORES[split])

264 translations in 497s
validation {'n': 75, 'chrf': 12.0, 'stopped': 1.0, 'distinct_outputs': 5, 'copies_of_training_targets': 0}
test {'n': 75, 'chrf': 11.6, 'stopped': 1.0, 'distinct_outputs': 3, 'copies_of_training_targets': 0}
ood {'n': 114, 'chrf': 12.4, 'stopped': 1.0, 'distinct_outputs': 8, 'copies_of_training_targets': 0}


## Cell 8 — Examples

Twelve random translations from validation, test and out-of-domain: encoder input, decoder output, reference.

In [9]:
# Cell 8 — Random examples

for x in random.Random(7).sample(RESULTS, 12):
    print(f"[{x['split']}] {x['persona']}")
    print(f"  encoder input : {x['persona']}: {x['source']}")
    print(f"  decoder output: {x['output']}" + ("" if x["stopped"] else "   [did not stop]"))
    print(f"  reference     : {x['references'][0]}\n")

[ood] don
  encoder input : don: Battery died, call you from home.
  decoder output: Pehle call karna hoga. Phir baad mein nahi.
  reference     : Battery khatam. Ghar se call karunga.

[test] jai_sholay
  encoder input : jai_sholay: I can accept a delay if you explain the reason and the new timeline.
  decoder output: Pehle sabki suno, phir decide karo.
  reference     : Deri theek hai, agar tum wajah aur nayi samay seema batao.

[ood] anthony_gonsalves
  encoder input : anthony_gonsalves: Lost my earphones on the bus.
  decoder output: Pehle check karna hoga, phir approve karna hoga.
  reference     : Earphones bus mein kho gaye, ab bus gaane sunegi!

[validation] don
  encoder input : don: The customer is confused because we gave them conflicting answers.
  decoder output: Pehle sabki suno. Phir decide karo.
  reference     : Customer confuse hai. Wajah? Hamare jawab aapas mein takra rahe the.

[validation] anthony_gonsalves
  encoder input : anthony_gonsalves: If you reach before m

## Cell 9 — Settings and every translation

In [10]:
# Cell 9 — Settings and records

print(json.dumps({"revisions": REV, "xattn_layers": XATTN_LAYERS, "lrs": LRS, "micro_batch": MICRO,
                  "accumulation": ACCUM, "kept_epoch": BEST_EPOCH, "log": LOG, "scores": SCORES}, indent=1))
print("\n----- RECORDS (JSON lines) -----")
for x in RESULTS:
    print(json.dumps(x, ensure_ascii=False))
print("----- END RECORDS -----")

{
 "revisions": {
  "answerdotai/ModernBERT-base": "8949b909ec900327062f0ebf497f51aef5e6f0c8",
  "google/gemma-3-4b-it": "093f9f388b31de276ce2de164bdc2081324b9767"
 },
 "xattn_layers": [
  3,
  7,
  11,
  15,
  19,
  23,
  27,
  31
 ],
 "lrs": {
  "encoder LoRA": 0.0001,
  "bridge": 0.0003,
  "cross-attention": 0.0003,
  "gates": 0.01,
  "decoder LoRA": 5e-05
 },
 "micro_batch": 4,
 "accumulation": 2,
 "kept_epoch": 2,
 "log": [
  {
   "epoch": 0,
   "validation_loss": 7.3383
  },
  {
   "epoch": 1,
   "train_loss": 4.3231,
   "validation_loss": 4.0494,
   "gates": [
    -0.043,
    -0.07,
    -0.07,
    -0.102,
    0.085,
    0.109,
    -0.105,
    0.272
   ]
  },
  {
   "epoch": 2,
   "train_loss": 2.8165,
   "validation_loss": 4.0234,
   "gates": [
    -0.04,
    -0.066,
    -0.081,
    -0.119,
    0.096,
    0.125,
    -0.12,
    0.253
   ]
  },
  {
   "epoch": 3,
   "train_loss": 1.8612,
   "validation_loss": 4.2228,
   "gates": [
    -0.032,
    -0.054,
    -0.074,
    -0.097,
  